# The Number Detective

**Programming Design Principles (5N2927) -- Skills Demo 1, 30%**
**Maths for IT (5N18396) -- Assignment 2, 30%**

Dundrum College of Further Education | Dublin and Dun Laoghaire ETB

---

In this skills demo you will build a collection of small programs that investigate numbers: how they are encoded, how to find them, and how to put them in order. Each exercise is a self-contained puzzle. Work through them at your own pace over the three days.

For each exercise, you are asked to write pseudocode, implement a function, and test it. The pseudocode and testing are not afterthoughts -- they carry real marks.

**What to submit:** This completed notebook with all cells run, plus any screen captures of your work.

## Assessment Criteria

| Criterion | Marks | What we are looking for |
|---|---|---|
| **Algorithm** | 8 | Pseudocode or flowchart for each exercise, plus a data dictionary |
| **Accurate Programming** | 16 | Working code with appropriate data types, functions, selection, iteration |
| **Documentation** | 4 | Comments, docstrings, clear variable names |
| **Testing** | 4 | Your own test cases with manual calculations to verify |

**Maths for IT** criteria are assessed through the mathematical content embedded in each exercise (number bases, sigma/pi notation, algorithm analysis).

---

## Helper Tools

The cell below provides some plotting and timing functions you can use throughout. Run it first.

In [ ]:
# === HELPER FUNCTIONS (provided -- just run this cell) ===

import time
import random
import matplotlib.pyplot as plt

def plot_comparisons(labels, values, title="Comparison Counts"):
    """Bar chart comparing counts (e.g., number of comparisons)."""
    plt.figure(figsize=(8, 4))
    colours = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f']
    plt.bar(labels, values, color=colours[:len(labels)], edgecolor='black')
    plt.ylabel('Count')
    plt.title(title)
    for i, v in enumerate(values):
        plt.text(i, v + max(values)*0.02, str(v), ha='center', fontsize=11)
    plt.tight_layout()
    plt.show()

def time_function(func, *args, repeats=1000):
    """Time a function call, averaged over many repeats. Returns seconds."""
    start = time.perf_counter()
    for _ in range(repeats):
        func(*args)
    elapsed = (time.perf_counter() - start) / repeats
    return elapsed

def plot_timing(labels, times, title="Execution Time"):
    """Bar chart comparing execution times."""
    plt.figure(figsize=(8, 4))
    colours = ['#4e79a7', '#f28e2b', '#e15759']
    plt.bar(labels, [t * 1000 for t in times], color=colours[:len(labels)], edgecolor='black')
    plt.ylabel('Time (milliseconds)')
    plt.title(title)
    for i, t in enumerate(times):
        plt.text(i, t*1000 + max(times)*1000*0.02, f"{t*1000:.4f} ms", ha='center', fontsize=10)
    plt.tight_layout()
    plt.show()

def scrambled_playlist(n=20):
    """Generate a shuffled list of n song titles (as numbered strings)."""
    songs = [f"Track {i+1:02d}" for i in range(n)]
    random.shuffle(songs)
    return songs

def show_sort_progress(items, title=""):
    """Print a compact view of a list during sorting."""
    display = [str(x) for x in items[:30]]
    if len(items) > 30:
        display.append("...")
    print(f"  {title}: [{', '.join(display)}]")

print("Helper functions loaded.")

---

## Day 1: Codes and Searches

---

### Exercise 1: The Secret Message

Someone has encoded a message as a list of numbers. Each number is the ASCII code of a character. Your job is to decode it.

But here is the twist: some of the codes are written in decimal, some in binary (prefixed with `0b`), and some in hexadecimal (prefixed with `0x`). You will need to handle all three.

**Background:** ASCII maps numbers to characters. `chr(72)` gives `'H'`, `chr(101)` gives `'e'`. Binary `0b1001000` is 72 in decimal. Hex `0x48` is also 72.

In [ ]:
# Here is the secret message (a mix of decimal, binary, and hex codes)
secret = [0x48, 101, 0b1101100, 0b1101100, 111, 0x2C, 32, 
          0b1110111, 0x6F, 114, 0b1101100, 100, 0x21]

# Quick demo of chr() and number bases
print("chr(72) =", chr(72))
print("0b1001000 =", 0b1001000)
print("0x48 =", 0x48)

**Your task:** Write a function `decode_message(codes)` that takes a list of integer codes and returns the decoded string.

Note: Python has already converted the binary and hex literals to integers for you in the list above. So `0x48` is just the integer 72 by the time your function sees it. The real exercise is understanding *why* these all represent the same number.

**Also:** Convert the number 42 to binary and hexadecimal by hand (show your working in comments), then verify with Python's `bin()` and `hex()` functions.

In [ ]:
# Pseudocode:
# For each code in the list of codes:
#     convert it to its character with chr()
#     add that character to a running string
# Return the finished string


In [ ]:
def decode_message(codes):
    """Decode a list of ASCII codes into a string."""
    message = ""
    for code in codes:
        message += chr(code)
    return message


In [ ]:
# Test your function
print(decode_message(secret))

# Manual conversions: show your working
# 42 in binary:
#   42 / 2 = 21 remainder 0
#   21 / 2 = 10 remainder 1
#   10 / 2 = 5  remainder 0
#   5  / 2 = 2  remainder 1
#   2  / 2 = 1  remainder 0
#   1  / 2 = 0  remainder 1
#   Reading the remainders bottom to top: 101010
#
# 42 in hex:
#   42 / 16 = 2 remainder 10 (which is "A" in hex)
#   2  / 16 = 0 remainder 2
#   Reading the remainders bottom to top: 2A

# Verify with Python
print("42 in binary:", bin(42))
print("42 in hex:", hex(42))


Now write your own secret message. Encode a short phrase as a mix of decimal, binary, and hex codes. Put the encoded version in a list and use your function to decode it.

In [ ]:
# Your secret message
my_message = [72, 0b1101001, 0x21]   # "Hi!" written as decimal, binary, and hex codes

# Decode it
print(decode_message(my_message))


---

### Exercise 2: The Guessing Game

Let's build a game where the computer tries to guess a number you are thinking of. The computer should use *binary search* to guess as efficiently as possible.

The idea: the computer guesses the middle of the remaining range. You tell it "higher" or "lower." It eliminates half the possibilities each time.

For this exercise, we will simulate the game automatically (the computer plays against itself) so we can count exactly how many guesses it takes.

In [ ]:
# Pseudocode:
# Keep a low bound, a high bound, and a guess counter starting at 0.
# While low <= high:
#     guess the midpoint of the current range, and count the guess
#     if the guess equals the target, stop and return the guess count
#     if the guess is too low, move low up past it
#     if the guess is too high, move high down past it


In [ ]:
def guessing_game(target, low, high):
    """
    Use binary search to find the target number.

    Returns:
        the number of guesses it took
    """
    guesses = 0
    while low <= high:
        guess = (low + high) // 2
        guesses += 1
        if guess == target:
            return guesses
        elif guess < target:
            low = guess + 1
        else:
            high = guess - 1
    return guesses


In [ ]:
# Test: guess a number between 1 and 100
print(guessing_game(73, 1, 100))

# Test: guess a number between 1 and 1,000,000
print(guessing_game(314159, 1, 1000000))


**Mathematical connection:** The maximum number of guesses for a range of size $n$ is $\lceil \log_2(n) \rceil$. Verify this: for $n = 100$, what is $\log_2(100)$? Does your function ever take more guesses than that?

In [ ]:
import math

# Verify the log2 relationship
for n in [100, 1000, 1000000]:
    print(f"log2({n}) = {math.log2(n):.2f}, ceiling = {math.ceil(math.log2(n))}")

# Run the game for EVERY number in 1-100 and find the maximum guesses:
worst = 0
for target in range(1, 101):
    guesses = guessing_game(target, 1, 100)
    if guesses > worst:
        worst = guesses
print("Worst case for range 1-100:", worst, "guesses")


Now let's *visualise* the efficiency. Run the guessing game for ranges of different sizes and plot how the number of guesses grows:

In [ ]:
# Compare guessing game efficiency across different range sizes
sizes = [10, 100, 1000, 10000, 100000]
max_guesses = []

for size in sizes:
    # Find the worst case: try every possible target
    worst = 0
    for target in range(1, size + 1):
        guesses = guessing_game(target, 1, size)
        if guesses > worst:
            worst = guesses
    max_guesses.append(worst)

plot_comparisons([str(s) for s in sizes], max_guesses, 
                 "Worst-Case Guesses vs Range Size")

---

## Day 2: Sums, Products, and Speed

---

### Exercise 3: Sigma and Pi

In mathematics, $\sum_{i=1}^{n} f(i)$ means "add up $f(i)$ for every $i$ from 1 to $n$." Similarly, $\prod_{i=1}^{n} f(i)$ means "multiply them all together."

These are just loops with accumulators. Let's make them into reusable functions.

In [ ]:
# Pseudocode:
# sigma(func, start, end):
#     total = 0
#     for i from start to end (inclusive): total += func(i)
#     return total
#
# pi_product(func, start, end):
#     product = 1
#     for i from start to end (inclusive): product *= func(i)
#     return product


In [ ]:
def sigma(func, start, end):
    """Compute the sum of func(i) for i from start to end (inclusive)."""
    total = 0
    for i in range(start, end + 1):
        total += func(i)
    return total

def pi_product(func, start, end):
    """Compute the product of func(i) for i from start to end (inclusive)."""
    product = 1
    for i in range(start, end + 1):
        product *= func(i)
    return product


In [ ]:
# Test cases -- verify by hand first, then check

# sigma(lambda i: i, 1, 5) should be 1+2+3+4+5 = 15
# sigma(lambda i: i**2, 1, 4) should be 1+4+9+16 = 30
# pi_product(lambda i: i, 1, 5) should be 5! = 120
print(sigma(lambda i: i, 1, 5))
print(sigma(lambda i: i**2, 1, 4))
print(pi_product(lambda i: i, 1, 5))


**Apply them:** Use your `sigma` function to compute:

1. $\sum_{i=1}^{100} i$ (the sum Gauss reportedly computed as a child; should be 5050)
2. $\sum_{i=1}^{10} \frac{1}{i}$ (the first 10 terms of the harmonic series)
3. Use `pi_product` to compute $10!$

In [ ]:
# Your calculations here
print("Sum 1 to 100:", sigma(lambda i: i, 1, 100), "expected 5050")
print("Harmonic sum (first 10 terms):", sigma(lambda i: 1 / i, 1, 10))
print("10!:", pi_product(lambda i: i, 1, 10))


---

### Exercise 4: Two Ways to Fibonacci

The Fibonacci sequence is 1, 1, 2, 3, 5, 8, 13, 21, ... where each number is the sum of the previous two.

There are (at least) two ways to compute the $n$-th Fibonacci number. The naive way recalculates the same values over and over. The clever way remembers what it has already computed.

Implement both and compare their speed.

In [ ]:
# Pseudocode for both approaches:
# fib_loop(n):
#     keep two running values a and b, starting at the first two Fibonacci numbers
#     step forward n-1 times, each time shifting (a, b) to (b, a+b)
#     return a
#
# fib_recursive(n):
#     if n is 1 or 2, return 1
#     otherwise return fib_recursive(n-1) + fib_recursive(n-2)


In [ ]:
def fib_loop(n):
    """Compute the n-th Fibonacci number using a simple loop."""
    a, b = 1, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return a

def fib_recursive(n):
    """Compute the n-th Fibonacci number using recursion (naive)."""
    if n <= 2:
        return 1
    return fib_recursive(n - 1) + fib_recursive(n - 2)


In [ ]:
# Test both: they should give the same answers
for i in [1, 5, 10, 15, 20]:
    print(f"fib({i}): loop={fib_loop(i)}, recursive={fib_recursive(i)}")


Now the interesting part: *time them*. Try computing `fib(30)` with each method. What happens?

In [ ]:
# Time both methods for increasing values of n
# WARNING: fib_recursive gets very slow! Don't go above 35 or so.

ns = [10, 15, 20, 25, 30]
loop_times = []
recursive_times = []

for n in ns:
    t_loop = time_function(fib_loop, n, repeats=100)
    t_rec = time_function(fib_recursive, n, repeats=10)  # fewer repeats -- it's slow!
    loop_times.append(t_loop)
    recursive_times.append(t_rec)
    print(f"n={n}: loop={t_loop*1000:.4f}ms, recursive={t_rec*1000:.4f}ms")

**Why is the recursive version so slow?** The recursive call for `fib(30)` ends up computing `fib(1)` and `fib(2)` millions of times. The loop version computes each value exactly once.

Write a brief explanation in your own words of why the two approaches have such different speeds:



---

## Day 3: Sorting it Out

---

### Exercise 5: Sort the Playlist

Your music player's playlist has been scrambled. Sort it back into order using **two different sorting algorithms** of your choice (bubble sort, insertion sort, or selection sort). Count how many comparisons each one makes.

In [ ]:
# Pseudocode for your two chosen algorithms:
# For each algorithm, walk through the same idea as the earlier notebooks:
# compare neighbouring elements, swap when they are out of order, and add
# one to a comparisons counter every time two elements are compared,
# whether or not that comparison leads to a swap.


In [ ]:
playlist = scrambled_playlist(20)
print("Scrambled:", playlist)

In [ ]:
def sort_method_1(items):
    """
    Sort using bubble sort.
    Returns: (sorted list, number of comparisons)
    """
    items = items.copy()
    comparisons = 0
    n = len(items)
    for i in range(n):
        swapped = False
        for j in range(n - 1 - i):
            comparisons += 1
            if items[j] > items[j + 1]:
                items[j], items[j + 1] = items[j + 1], items[j]
                swapped = True
        if not swapped:
            break
    return items, comparisons

def sort_method_2(items):
    """
    Sort using insertion sort.
    Returns: (sorted list, number of comparisons)
    """
    items = items.copy()
    comparisons = 0
    for i in range(1, len(items)):
        current = items[i]
        j = i - 1
        while j >= 0:
            comparisons += 1
            if items[j] <= current:
                break
            items[j + 1] = items[j]
            j -= 1
        items[j + 1] = current
    return items, comparisons


In [ ]:
# Test both on the same scrambled playlist
sorted1, comps1 = sort_method_1(playlist)
sorted2, comps2 = sort_method_2(playlist)

print("Method 1 sorted correctly:", sorted1 == sorted(playlist))
print("Method 2 sorted correctly:", sorted2 == sorted(playlist))

plot_comparisons(["Method 1", "Method 2"], [comps1, comps2],
                 "Sorting Comparisons (20 items)")

**Scale it up:** How do the comparison counts grow as the playlist gets bigger? Test with playlists of 10, 20, 50, and 100 items.

In [ ]:
# Compare at different sizes
sizes = [10, 20, 50, 100]
m1_counts = []
m2_counts = []

for size in sizes:
    data = scrambled_playlist(size)
    _, c1 = sort_method_1(data)
    _, c2 = sort_method_2(data)
    m1_counts.append(c1)
    m2_counts.append(c2)

# Plot side by side
fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(sizes))
width = 0.35
ax.bar([i - width/2 for i in x], m1_counts, width, label='Method 1', color='#4e79a7')
ax.bar([i + width/2 for i in x], m2_counts, width, label='Method 2', color='#f28e2b')
ax.set_xticks(list(x))
ax.set_xticklabels([str(s) for s in sizes])
ax.set_xlabel('Playlist Size')
ax.set_ylabel('Comparisons')
ax.set_title('Sorting Comparisons by Playlist Size')
ax.legend()
plt.tight_layout()
plt.show()

**Reflect:** Both algorithms are $O(n^2)$ in the worst case, meaning comparisons grow roughly as the square of the input size. Does your data confirm this? If the playlist is *already sorted*, does one algorithm do better than the other?

In [ ]:
# Test with an already-sorted playlist
sorted_playlist = [f"Track {i+1:02d}" for i in range(50)]
_, c1_sorted = sort_method_1(sorted_playlist)
_, c2_sorted = sort_method_2(sorted_playlist)
print("Already sorted -- Method 1:", c1_sorted, "comparisons")
print("Already sorted -- Method 2:", c2_sorted, "comparisons")

---

### Data Dictionary

List all the key variables and functions you created, with their types and purposes:

| Name | Type | Purpose |
|---|---|---|
| | | |
| | | |



---

### Final Reflection

Write a few sentences: which exercise taught you the most? What was the most surprising result?

